# Assignment 10

In [ ]:
import numpy as np
import cv2
import mediapipe as mp
import pandas as pd
import random
import os
import time
import json
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import torch
import torch.nn as nn
import torch.optim as optim

if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
    print("Apple GPU")
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU / AMD ROCm GPU
    print("Nvidia/AMD GPU")
else:
    device = torch.device("cpu")
    print("CPU")


random_seed = 42

### Compare Kinect to Mediapipe

**OBS!** Work In Progress

In [ ]:

# def euclidean(x1, y1, z1, x2, y2, z2):
#     return np.sqrt((x1-x2)**2 + (y1-y2)**2 + (z1-z2)**2)

# def euclidean(x1, y1, x2, y2):
#     return np.sqrt((x1-x2)**2 + (y1-y2)**2)

def euclidean(x1, x2):
    return np.abs((x1-x2))

video_tags = {"A":159, "B":22}
base_dir_mediapipe = "../../MainProject/Assignment10/data/csv_of_all_videos/"
base_dir_kinect = "../../MainProject/Assignment9/data/kinect_good_preprocessed/"

differences = []

for letter, number in video_tags.items():
    for i in range(1, number + 1):
        mediapipe_dir = base_dir_mediapipe + f"{letter}{i}_mediapipe.csv"
        kinect_dir = base_dir_kinect + f"{letter}{i}_kinect.csv"

        df_mediapipe = pd.read_csv(mediapipe_dir)
        df_kinect = pd.read_csv(kinect_dir)

        # Kinect files have a column named " head_x" instead of "head_x" for some reason...
        df_kinect = df_kinect.rename(columns={" head_x":"head_x"})

        # Only keep frames that exist in both mediapipe and kinect DataFrames
        frames = df_kinect["FrameNo"].values
        df_mediapipe = df_mediapipe.loc[df_mediapipe.index.isin(frames)]
        df_mediapipe = df_mediapipe.reset_index(drop=True)

        # Need this since mediapipe sets origin in upper left corner
        for column in [c for c in df_kinect.columns if c.endswith("_y")]:
            df_mediapipe[column] = 1 - df_mediapipe[column]

        # Set origin at the midpoint of the hips for both datasets
        for axis in ("_x", "_y", "_z"):

            # Setup kinect
            left_hip_kinect = df_kinect[f"left_hip{axis}"]
            right_hip_kinect = df_kinect[f"right_hip{axis}"]
            hip_mid_kinect = (left_hip_kinect + right_hip_kinect) / 2

            # Setup mediapipe
            left_hip_mediapipe = df_mediapipe[f"left_hip{axis}"]
            right_hip_mediapipe = df_mediapipe[f"right_hip{axis}"]
            hip_mid_mediapipe = (left_hip_mediapipe + right_hip_mediapipe) / 2

            right_hand_kinect = df_kinect[f"right_hand{axis}"]
            right_hand_mediapipe = df_mediapipe[f"right_hand{axis}"]
            scaling_factor = right_hand_kinect / right_hand_mediapipe

            axis_cols = [c for c in df_kinect.columns if c.endswith(axis)]
            for col in axis_cols:
                df_kinect[col] -= hip_mid_kinect
                df_mediapipe[col] -= hip_mid_mediapipe
                df_mediapipe[col] *= scaling_factor

        df_kinect = df_kinect.drop("FrameNo", axis=1)
        df_mediapipe = df_mediapipe.drop("FrameNo", axis=1)

        df_difference = np.abs(df_kinect - df_mediapipe)
        differences += [df_difference]

        df_percentages = np.abs(df_difference / df_kinect)*100
        break
    break

# print(f"{'Node':<25} {'Kinect (m)':>10} {'MediaPipe (m)':>15} {'Error (m)':>12} {'Error (%)':>10}")
# print("-" * 75)
# for column in df_difference.columns:
#     print(f"{column:<25} {df_kinect[column][0]:>10.3f} {df_mediapipe[column][0]:>15.3f} {df_difference[column][0]:>12.3f} {df_percentages[column][0]:>10.2f}")


### Take the function to load the champion model's metadata

In [ ]:
# Base paths
base_model_dir = "../../MainProject/models"
candidates_dir = os.path.join(base_model_dir, "candidates")
champion_dir = os.path.join(base_model_dir, "champion")
metadata_dir = os.path.join(base_model_dir, "metadata")

# Create folders if they don't exist
os.makedirs(candidates_dir, exist_ok=True)
os.makedirs(champion_dir, exist_ok=True)
os.makedirs(metadata_dir, exist_ok=True)


def load_champion_info():
    path = os.path.join(metadata_dir, "champion_info.json")

    if not os.path.exists(path):
        return None

    try:
        with open(path, "r") as f:
            return json.load(f)
    except:
        return None

### Hugo's data functions and data splitting

In [ ]:
def split_csvfiles(datafolder, random_seed):
    csv_files = []
    for f in os.listdir(datafolder):
        if f.endswith(".csv"):
            csv_files.append(f)
    
    random.seed(random_seed)
    random.shuffle(csv_files)

    # Train/Validation/Test with proportions 80/10/10
    train_n = int(len(csv_files) * 0.8)
    val_n = int(len(csv_files) * 0.1)
    test_n = len(csv_files) - train_n - val_n

    # Split
    train_files = csv_files[:train_n]
    val_files = csv_files[train_n: train_n + val_n]
    test_files = csv_files[train_n + test_n:]

    return train_files, val_files, test_files

def load(files, data_dir):
    dataframes = []

    for f in files:
        path = os.path.join(data_dir, f)   
        df = pd.read_csv(path)           
        dataframes.append(df)             

    combined = pd.concat(dataframes, ignore_index=True)  # combine all

    return combined


def input_target_split(dataframe):
    input_cols = []
    target_cols = []
    for c in dataframe.columns:
        if c.endswith("_x") or c.endswith("_y"):
            input_cols.append(c)
        if c.endswith("_z"):
            target_cols.append(c)
    
    input_data = dataframe[input_cols]
    target_data = dataframe[target_cols]
    return input_data, target_data

In [ ]:
datafolder = "../../MainProject/Assignment9/data/kinect_good_preprocessed"

train_files, val_files, test_files = split_csvfiles(datafolder, random_seed)

train_data = load(train_files, datafolder)
val_data = load(val_files, datafolder)
test_data = load(test_files, datafolder)

# val_data has one extra column "head_x" appended at the end for some reason
cols = [val_data.columns[i] for i in range(40)]
val_data = val_data[cols]

x_train, y_train = input_target_split(train_data)
x_val, y_val = input_target_split(val_data)
x_test, y_test = input_target_split(test_data)

x_train = torch.tensor(x_train.values, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train.values, dtype=torch.float32).to(device)
x_val = torch.tensor(x_val.values, dtype=torch.float32).to(device)
y_val = torch.tensor(y_val.values, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test.values, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test.values, dtype=torch.float32).to(device)

### Train the best model

In [ ]:
def init_weights(m):
        if isinstance(m, nn.Linear):
            #nn.init.xavier_uniform_(m.weight)  # good for Tanh
            nn.init.kaiming_uniform_(m.weight) # good for ReLU
            nn.init.zeros_(m.bias)

    
class ZPredictor(nn.Module):
    def __init__(self, hidden_layers: list, activation=nn.ReLU(), dropout=0.0):
        super().__init__()
        
        layers = []
        input_size = 26  # 13 joints x 2 (x, y)
        
        for hidden_size in hidden_layers:
            layers.append(nn.Linear(input_size, hidden_size))
            layers.append(activation)
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            input_size = hidden_size
        
        layers.append(nn.Linear(input_size, 13))  # 13 joints z output
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

In [ ]:
params = load_champion_info()["hyperparameters"]

if params["activation"] == "relu":
    activation = nn.ReLU()
elif params["activation"] == "leaky_relu":
    activation = nn.LeakyReLU()
elif params["activation"] == "gelu":
    activation = nn.GELU()
elif params["activation"] == "tanh":
    activation = nn.Tanh()
else:
    activation = nn.ReLU()

champion = ZPredictor(params["hidden_layers"], activation, params["dropout"]).to(device)
champion.apply(init_weights)
optimizer = optim.Adam(champion.parameters(), lr=params["learning_rate"])
best_val_loss = float("inf")
val_loss = float("inf")
loss_fn = nn.MSELoss()

start_time = time.perf_counter()
epochs_no_improve = 0

for epoch in range(params["epochs"]):

    # Train step
    champion.train()
    optimizer.zero_grad()
    predictions = champion(x_train)
    train_loss = loss_fn(predictions, y_train)
    train_loss.backward()
    optimizer.step()
    
    val_predictions = champion(x_val)
    val_loss = loss_fn(val_predictions, y_val)

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        # mlflow.pytorch.log_model(model, name="best_model")
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= params["patience"]:
        break


### Predict the Z values from the dataset

In [ ]:
differences = []

y_predictions = champion(x_test)

abs_e = torch.abs(y_predictions - y_test).cpu().detach().numpy()

print("Average absolute error (MAE):")
for i in range(len(abs_e[0])):
    print(f"\t{df_kinect.columns[i * 3 + 2]:>20}: {abs_e[:, i].mean() * 100:.3f} cm")